In [293]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-07-03
Last modified on 2024-07-03
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, 
@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md
'''

'\nCreated on 2024-07-03\nLast modified on 2024-07-03\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, \n@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md\n'

**Índice de contenidos**

[Requerimientos]()

[Funciones]()

[Parámetros]()

[Ejecución principal]()


[1. Elementos generales]()

- [1.1. Generación de la matrix MITRE]()

- [1.2. Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada]()

- [1.3. Obtención de las TTP disponbles en Cyber Proof con regla de detección]()

[2. Tablas informativas]()

- [2.1. Técnicas]()

- [2.2. Tácticas]()

- [2.3. Data sources]()

- [2.4. Plataformas]()

- [2.5. Grupos]()

- [2.6. Software]()

[3. Relaciones]()

- [3.1. Relación técnicas - tácticas]()

- [3.2. Relación técnicas - data sources]()

- [3.3. Relación técnicas - plataformas]()

- [3.4. Relación técnicas - grupos]()

- [3.5. Relación técnicas - software]()

- [3.6. Relación grupos - software]()

[4. Disponibilidad de reglas por tipología]()

- [4.1. Reglas disponibles por técnica]()

- [4.2. Reglas disponibles por técnica y táctica]()

- [4.3. Reglas disponibles por técnica y data source]()

- [4.4. Reglas disponibles por técnica y plataforma]()

- [4.5. Reglas disponibles por técnica y grupo]()

- [4.6. Reglas disponibles por técnica y software]()

#### **Requerimientos**

In [294]:
from stix2 import Filter, MemoryStore
import stix2
import requests

import os
import pandas as pd

import datetime
import csv

#### **Funciones**

**Generales**

In [295]:
def create_output_folder(path):
    '''
    Función encargada para crear el directorio facilitado en caso de no existir previamente.
    '''
    if not os.path.exists(path):
        os.makedirs(path)

In [296]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [297]:
def get_unique_ttps_generated(paths):
    '''
    Función encargada de retornar una lista de id de TTP's únicas generadas. Esta función recibirá una ruta generada previamente en la estructura de carpetas de guardado. 
    '''
    sub_folders = set()
    for path in paths:
        dir, file = os.path.split(path)
        last_sub_folder = os.path.basename(dir).strip()
        sub_folders.add(last_sub_folder)
    return list(sub_folders)

In [298]:
def unique_list(series):
    '''
    Función para devolver sólo ítems únicos a partir de una lista.
    '''
    return list(set(series))

In [299]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verificar si la solicitud fue exitosa
        stix_json = response.json()
        
        if "objects" in stix_json:
            return MemoryStore(stix_data=stix_json["objects"])
        else:
            raise ValueError("JSON no contiene la clave 'objects'")
            
    except requests.RequestException as e:
        print(f"Error al obtener datos de {matrix}-attack: {e}")
        return None

In [300]:
def save_df_as_csv(df, name, matrix, aux_folder=''):
    '''
    Función encargada de guardar un dataframe pasado como argumento como csv.
    '''
    now = datetime.datetime.now()
    if not os.path.exists(os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder)):
        os.makedirs(os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder))
    # file_to_save = os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder, (name+ '_'+ now.strftime('%d%m%Y_%H%M')+'h.csv'))
    file_to_save = os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder, (name + '.csv'))
    df.to_csv(file_to_save, sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)
    # print('Archivo guardado correctamente '+ name + '_' +now.strftime('%d%m%Y_%H%M')+'h.csv')
    print('Archivo guardado correctamente '+ (name + '.csv'))

In [301]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [302]:
def get_CP_ttps_with_rules(mitre_matrix_str, way='file'):
    '''
    Función encargada de obtener el listado de TTP's para las que desde CP disponemos de reglas de detección. Para la ejecutarla correctamente debe de haberse ejecutado previamente el código del bloque get_rules_and_classify_by_ttp. Hay dos formas de evaluar las técnicas disponibles y son gestionadas por el parámetro "way". Por defecto el modo es "file", lo que nos indica que se va a evaluar uno de los ficheros generados de resumen de reglas por TTP clasificadas. En el caso de que el parámetro "way" recoja el valor "path", lo que hará es identificar las reglas mediante la exploración de subdirectorios. 
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix_str)
        
        if not os.path.exists(main_path):
            raise ValueError(f'No existe la path: {main_path}')
        
        if way == 'file':
            file_path = os.path.join(main_path, f'{mitre_matrix_str}-ttp_all_classified_rules.csv')  # Corrección aquí
            file = pd.read_csv(file_path, sep=';')
            file = file[file['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
            CP_techniques = file['ttp'].unique().tolist()  # Convertimos en lista de items únicos la columna que informa de la ttp
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)  # Ordenamos por longitud de caracteres para que cuando sea utilizada esta lista primero se evalúen las subtécnicas.

        elif way == 'path':
            CP_techniques = get_unique_ttps_generated(get_list_of_files_sub(main_path))
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)
            CP_techniques = [ttp for ttp in CP_techniques if ttp.startswith('T')]
            CP_techniques = [ttp for ttp in CP_techniques if ttp != 'T0000']

        else:
            CP_techniques = []
            
    except ValueError as e:
        print(f'{e}')
        CP_techniques = []
    
    return CP_techniques

In [303]:
def get_CP_and_NOCP_techniques_from_df(cp_techniques_list, df):
    '''
    Función encargada de filtrar un df dado el cual contiene el campo techniques_ID por las técnicas disponibles en CP (estas son facilitadas como parámetro en forma de lista). 
    '''
    cp_df = pd.DataFrame()
    nocp_df = pd.DataFrame() 
    try:
        cp_df = df[df['technique_ID'].isin(cp_techniques_list)]
        nocp_df = df[~df['technique_ID'].isin(cp_techniques_list)]
    except:
        print('No se ha podido ejecutar la operación.')
    return cp_df, nocp_df

**Obtención de información**

In [304]:
def get_techniques_information(matrix_store, cp_techniques_list, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a la técnicas (ID, nombre, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    techniques_data = []
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    for tech in techniques:
        deprecated = False
        revoked = False
        description = ''
        if 'x_mitre_deprecated' in tech: 
            deprecated = tech['x_mitre_deprecated']
        if 'revoked' in tech: 
            revoked = tech['revoked']
        if 'description' in tech: 
            description = tech['description']
        techniques_data.append({
            "technique_ID": tech['external_references'][0]['external_id'],
            "technique": tech['name'],
            "technique_url": tech['external_references'][0]['url'],
            "technique_description":description,
            "technique_deprecated": deprecated,
            "technique_revoked": revoked,
            "matrix_domains": tech['x_mitre_domains'] # pendiente chequear
        })
        
    MITRE_techniques_df = pd.DataFrame(techniques_data)

    if revoked_deprecated:
        MITRE_techniques_df = MITRE_techniques_df[(MITRE_techniques_df['technique_deprecated']!=True)&(MITRE_techniques_df['technique_revoked']!=True)]

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_df, NOCP_techniques_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_df)

    return MITRE_techniques_df, CP_techniques_df, NOCP_techniques_df

In [305]:
def get_tactics_information(matrix_store):
    '''
    Función que retorna el dataframe con la información relativa a la tácticas (ID, nombre, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    tactics_data = []
    tactics = matrix_store.query([Filter('type', '=', 'x-mitre-tactic')])
    for tact in tactics:
        tactics_data.append({
            "tactic_ID": tact['external_references'][0]['external_id'],
            "tactic": tact['name'],
            "tactic_url": tact['external_references'][0]['url'],
            "tactic_description":tact['description'],
            "matrix_domains": tact['x_mitre_domains'] # pendiente chequear
        })
    MITRE_tactics_df = pd.DataFrame(tactics_data)
    return MITRE_tactics_df

In [306]:
def get_datasources_information(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a los data sources (ID, nombre, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    datasoruces_data = []
    data_sources = matrix_store.query([Filter('type', '=', 'x-mitre-data-source')])
    for ds in data_sources:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in ds: 
            deprecated = ds['x_mitre_deprecated']
        if 'revoked' in ds: 
            revoked = ds['revoked']

        datasoruces_data.append({
            "datasource_ID": ds['external_references'][0]['external_id'],
            "datasource": ds['name'],
            "datasource_url": ds['external_references'][0]['url'],
            "datasource_description":ds['description'],
            "datasource_deprecated": deprecated,
            "datasource_revoked": revoked,
            "matrix_domains": ds['x_mitre_domains'] # pendiente chequear
        })

    MITRE_datasources_df = pd.DataFrame(datasoruces_data)
    if revoked_deprecated:
        MITRE_datasources_df = MITRE_datasources_df[(MITRE_datasources_df['datasource_deprecated']!=True)&(MITRE_datasources_df['datasource_revoked']!=True)]
    return MITRE_datasources_df

In [307]:
def get_platforms_information(matrix_store):
    '''
    Función que retorna el dataframe con la información relativa a las plataformas. Únicamente está conformado por una columna con el nombre de la misma ya que las plataformas no se obtienen de una tipología propia de la matriz, se obtienen como parámetro de las técnicas.
    '''
    platforms_from_tech = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    platforms_data = []
    for technique in platforms_from_tech:
        platform_name = 'None'
        if 'x_mitre_data_sources' in technique: 
            try: 
                platform_name = technique['x_mitre_platforms']
            except:
                platform_name = 'None'

        platforms_data.append({
            "platform": platform_name,
        })
    # Generamos df a partir de los datos recopilados
    MITRE_platforms_df = pd.DataFrame(platforms_data)
    MITRE_platforms_df = MITRE_platforms_df.explode('platform')
    MITRE_platforms_df = MITRE_platforms_df.drop_duplicates()
    MITRE_platforms_df = MITRE_platforms_df.sort_values(by='platform')
    MITRE_platforms_df = MITRE_platforms_df.reset_index(drop=True)
    return MITRE_platforms_df

In [308]:
def get_groups_information(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a los grupos (ID, nombre, alias, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])
    groups_data = []
    for group in groups:
        aliases = ''
        description =''
        deprecated = False
        revoked = False
        if 'aliases' in group: 
            aliases = group['aliases']
        if 'description' in group: 
            description = group['description']
        if 'x_mitre_deprecated' in group: 
            deprecated = group['x_mitre_deprecated']
        if 'revoked' in group: 
            revoked = group['revoked']

        groups_data.append({
            "group_ID": group['external_references'][0]['external_id'],
            "group": group['name'],
            "group_aliases": aliases,
            "group_url": group['external_references'][0]['url'],
            "group_description": description,
            "group_deprecated": deprecated,
            "group_revoked": revoked,
            "matrix_domains": group['x_mitre_domains']
        })


    # Generamos df a partir de los datos recopilados
    MITRE_groups_df = pd.DataFrame(groups_data)
    if revoked_deprecated:
        MITRE_groups_df = MITRE_groups_df[(MITRE_groups_df['group_deprecated']!=True)&(MITRE_groups_df['group_revoked']!=True)]

    MITRE_groups_df = MITRE_groups_df.sort_values(by='group_ID')
    MITRE_groups_df = MITRE_groups_df.reset_index(drop=True)
    return MITRE_groups_df

In [309]:
def get_software_information(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a los tipos de software (ID, nombre, alias, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    software_data = []
    for sw in software:
        description =''
        deprecated = False
        revoked = False
        if 'aliases' in sw: 
            aliases = sw['aliases']
        if 'description' in sw: 
            description = sw['description']
        if 'x_mitre_deprecated' in sw: 
            deprecated = sw['x_mitre_deprecated']
        if 'revoked' in sw: 
            revoked = sw['revoked']

        software_data.append({
            "software_ID": sw['external_references'][0]['external_id'],
            "software": sw['name'],
            "software_type": sw['type'],
            "software_url": sw['external_references'][0]['url'],
            "software_description": description,
            "software_deprecated": deprecated,
            "software_revoked": revoked,
            "matrix_domains": sw['x_mitre_domains']
        })


    # Generamos df a partir de los datos recopilados
    MITRE_software_df = pd.DataFrame(software_data)
    revoked_deprecated = True
    if revoked_deprecated:
        MITRE_software_df = MITRE_software_df[(MITRE_software_df['software_deprecated']!=True)&(MITRE_software_df['software_revoked']!=True)]

    MITRE_software_df = MITRE_software_df.sort_values(by='software_ID')
    MITRE_software_df = MITRE_software_df.reset_index(drop=True)
    return MITRE_software_df

**Generación de relaciones**

In [310]:
def get_techniques_tactics_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y tácticas (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    # Obtenemos las tácticas de la matriz
    tactics = matrix_store.query([
        Filter('type', '=', 'x-mitre-tactic')
    ])
    
    # Mediante diccionarios mapeamos la información que necesitamos
    technique_dict = {
        tech['external_references'][0]['external_id']: {
            'name': tech['name'],
            'deprecated': tech.get('x_mitre_deprecated', False),
            'revoked': tech.get('revoked', False)
        }
        for tech in techniques
        if 'external_references' in tech and len(tech['external_references']) > 0
    }
    tactic_dict = {
        tac['external_references'][0]['external_id']: tac['name']
        for tac in tactics
        if 'external_references' in tac and len(tac['external_references']) > 0
    }
    # Lista para almacenar las relaciones
    data = []

    # Recorrer todas las técnicas para encontrar sus relaciones con tácticas
    for technique in techniques:
        if 'kill_chain_phases' in technique:
            for phase in technique['kill_chain_phases']:
                # Cada fase representa una relación técnica-táctica
                tactic_shortname = phase['phase_name']
                external_id = technique['external_references'][0]['external_id'] if 'external_references' in technique and len(technique['external_references']) > 0 else None
                
                # Buscar el ID externo de la táctica correspondiente
                tactic_id = next((tac['external_references'][0]['external_id'] for tac in tactics if tac['x_mitre_shortname'] == tactic_shortname), None)
                # Si no hubiera ninguna relación
                if not external_id or not tactic_id:
                    continue
                # Agregamos la relación a la lista
                data.append({
                    "technique_ID": external_id,
                    "technique": technique_dict[external_id]['name'],
                    "tactic_ID": tactic_id,
                    "tactic": tactic_dict[tactic_id],
                    "technique_deprecated": technique_dict[external_id]['deprecated'],
                    "technique_revoked": technique_dict[external_id]['revoked']
                })
    
    # Creamos el dataframe final
    techniques_tactics_df = pd.DataFrame(data)
    #Filtramos técnicas deprecadas
    techniques_tactics_NN_df = techniques_tactics_df[(techniques_tactics_df['technique_deprecated']!=True)&(techniques_tactics_df['technique_revoked']!=True)]

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_tactics_NN_df = techniques_tactics_NN_df[['technique_ID', 'technique', 'tactic_ID', 'tactic']] # Filtramos las columnas que necesitamos
    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_tactics_NN_df)


    # Para finalizar agregamos por técnica y táctica con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y tácticas en una relación 1:N
    MITRE_technique_tactics_1N_df = MITRE_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    CP_technique_tactics_1N_df = CP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    NOCP_technique_tactics_1N_df = NOCP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })


    # Generamos el df compuesto por tecnicas y tácticas en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [311]:
def get_techniques_datasources_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y data sources (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_ds_name = ''
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique: 
            ttp_ds_name = technique['x_mitre_data_sources']
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "data_source": ttp_ds_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)
    # Transformaciones de la tabla de relaciones de técnicas
    techniques_df = techniques_df.explode('data_source')
    techniques_df['data_source'] = techniques_df['data_source'].str.split(':').str[0] # El datasource al que aplica cada 
    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'data_source']]

    # Obtenemos los data sources de la matriz
    data_sources = matrix_store.query([
        Filter('type', '=', 'x-mitre-data-source')
    ])

    ds_data = []

    for data_source in data_sources:
        ds_id = ''
        ds_name = ''
        ds_revoked = ''
        ds_deprecated = ''
        if 'external_references' in data_source:
            ds_id = data_source['external_references'][0]['external_id']
        if 'name' in data_source:
            ds_name = data_source['name']
        if 'revoked' in data_source:
            ds_revoked = data_source['revoked']
        if 'x_mitre_deprecated' in data_source:
            ds_deprecated = data_source['x_mitre_deprecated']

        ds_data.append({
            "data_source_ID": ds_id,
            "data_source": ds_name,
            "data_source_deprecated": ds_deprecated,
            "data_source_revoked": ds_revoked
        })
    # Generamos df a partir de los datos recopilados
    data_sources_df = pd.DataFrame(ds_data)
    # Transformaciones de la tabla de relaciones de data sources
    data_sources_df = data_sources_df[(data_sources_df['data_source_deprecated']!=True)&(data_sources_df['data_source_revoked']!=True)]
    data_sources_df = data_sources_df[['data_source_ID', 'data_source']] 
    data_sources_df = data_sources_df.sort_values(by='data_source_ID')

    # Unimos las tablas
    techniques_data_sources_df = pd.merge(techniques_df, data_sources_df, on='data_source', how='left')
    techniques_data_sources_df = techniques_data_sources_df.drop_duplicates()

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y data sources en una relación N:N
    MITRE_techniques_datasources_NN_df = techniques_data_sources_df[['technique_ID', 'technique', 'data_source_ID', 'data_source']]
    MITRE_techniques_datasources_NN_df = MITRE_techniques_datasources_NN_df.sort_values(by='technique_ID').reset_index(drop=True)
    MITRE_techniques_datasources_NN_df

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_datasources_NN_df)


    # Para finalizar agregamos por técnica y data sources con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y data sources en una relación 1:N
    MITRE_technique_datasources_1N_df = MITRE_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    CP_technique_datasources_1N_df = CP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    NOCP_technique_datasources_1N_df = NOCP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })

    # Generamos el df compuesto por tecnicas y data sources en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [312]:
def get_techniques_platforms_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y plataformas (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_platform_name = 'None'
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique:
            try: 
                ttp_platform_name = technique['x_mitre_platforms']
            except:
                ttp_platform_name = 'None'
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "platform": ttp_platform_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)

    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'platform']]
    techniques_df = techniques_df.explode('platform')
    MITRE_techniques_platforms_NN_df = techniques_df.drop_duplicates()
    MITRE_techniques_platforms_NN_df = MITRE_techniques_platforms_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_platforms_NN_df)


    # Para finalizar agregamos por técnica y plataformas con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y plataformas en una relación 1:N
    MITRE_technique_platforms_1N_df = MITRE_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    CP_technique_platforms_1N_df = CP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    NOCP_technique_platforms_1N_df = NOCP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_platform_N1_df = MITRE_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_platform_N1_df = CP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_platform_N1_df = NOCP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df

In [313]:
def get_techniques_groups_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y grupos (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Filtramos los objetos de tipo intrusion-set (grupos), attack-pattern (técnicas) y relationship (relaciones)
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])

    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # En 2 diccionarios mapeamos los IDs y nombres
    group_id_map = {g['id']: next((ref['external_id'] for ref in g['external_references'] if ref['source_name'] == 'mitre-attack'), None) for g in groups}
    technique_id_map = {t['id']: {
            'technique_id': next((ref['external_id'] for ref in t['external_references'] if ref['source_name'] == 'mitre-attack'), None),
            'technique_name': t['name']
        } for t in techniques}

    # Creamos un diccionario para mapear los grupos y las técnicas que usan mediante las relaciones
    group_techniques = []

    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'].startswith('intrusion-set') and rel['target_ref'].startswith('attack-pattern'):
            group_id = rel['source_ref']
            technique_id = rel['target_ref']

            group_abbrev_id = group_id_map.get(group_id)
            technique_info = technique_id_map.get(technique_id)
            group_name = next((g['name'] for g in groups if g['id'] == group_id), None)

            if group_abbrev_id and technique_info and group_name:
                group_techniques.append({
                    'group_ID': group_abbrev_id,
                    'group': group_name,
                    'technique_ID': technique_info['technique_id'],
                    'technique': technique_info['technique_name']
                })

    # Crear un dataframe con los resultados
    df = pd.DataFrame(group_techniques, columns=['group_ID', 'group', 'technique_ID', 'technique'])

    MITRE_techniques_groups_NN_df = df.drop_duplicates()
    MITRE_techniques_groups_NN_df = MITRE_techniques_groups_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_groups_NN_df)


    # Para finalizar agregamos por técnica y grupos con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y grupos en una relación 1:N
    MITRE_technique_groups_1N_df = MITRE_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })
    CP_technique_groups_1N_df = CP_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })
    NOCP_technique_groups_1N_df = NOCP_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_group_N1_df = MITRE_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_group_N1_df = CP_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_group_N1_df = NOCP_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_groups_NN_df, CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df, MITRE_technique_groups_1N_df, CP_technique_groups_1N_df, NOCP_technique_groups_1N_df, MITRE_techniques_group_N1_df, CP_techniques_group_N1_df, NOCP_techniques_group_N1_df

In [314]:
def get_techniques_software_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y software (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Filtrar técnicas y software
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    software = matrix_store.query([Filter('type', '=', 'malware')]) + matrix_store.query([Filter('type', '=', 'tool')])
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # Crear diccionarios para los IDs y nombres de técnicas y software
    technique_id_to_name = {}
    software_id_to_name = {}

    for tech in techniques:
        for ref in tech.get('external_references', []):
            if 'external_id' in ref:
                technique_id_to_name[tech['id']] = {
                    'external_id': ref['external_id'],
                    'name': tech['name'],
                    'x_mitre_deprecated': tech.get('x_mitre_deprecated', False),
                    'revoked': tech.get('revoked', False)
                }
                break

    for sw in software:
        for ref in sw.get('external_references', []):
            if 'external_id' in ref:
                software_id_to_name[sw['id']] = {
                    'external_id': ref['external_id'],
                    'name': sw['name']
                }
                break

    # Extraer las relaciones entre técnicas y software
    techniques_software_data = []
    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'] in software_id_to_name and rel['target_ref'] in technique_id_to_name:
            software_info = software_id_to_name[rel['source_ref']]
            technique_info = technique_id_to_name[rel['target_ref']]
            techniques_software_data.append({
                'software_ID': software_info['external_id'],
                'software': software_info['name'],
                'technique_ID': technique_info['external_id'],
                'technique': technique_info['name'],
                'technique_deprecated': technique_info['x_mitre_deprecated'],
                'technique_revoked': technique_info['revoked']
            })

    techniques_software_df = pd.DataFrame(techniques_software_data)

    # Transformaciones de la tabla de relaciones de data sources
    techniques_software_df = techniques_software_df[(techniques_software_df['technique_deprecated']!=True)&(techniques_software_df['technique_revoked']!=True)]
    techniques_software_df = techniques_software_df.drop_duplicates()

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_software_NN_df = techniques_software_df[['technique_ID', 'technique', 'software_ID', 'software']]
    MITRE_techniques_software_NN_df = MITRE_techniques_software_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_software_NN_df, NOCP_techniques_software_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_software_NN_df)

    # Para finalizar agregamos por técnica y software con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y software en una relación 1:N
    MITRE_technique_software_1N_df = MITRE_techniques_software_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })
    CP_technique_software_1N_df = CP_techniques_software_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })
    NOCP_technique_software_1N_df = NOCP_techniques_software_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })

    # Generamos el df compuesto por tecnicas y software en una relación N:1
    MITRE_techniques_software_N1_df = MITRE_techniques_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_software_N1_df = CP_techniques_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_software_N1_df = NOCP_techniques_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_software_NN_df, CP_techniques_software_NN_df, NOCP_techniques_software_NN_df, MITRE_technique_software_1N_df, CP_technique_software_1N_df, NOCP_technique_software_1N_df, MITRE_techniques_software_N1_df, CP_techniques_software_N1_df, NOCP_techniques_software_N1_df

In [315]:
def get_groups_software_relationships(matrix_store):
    '''
    Función que retorna las relaciones entre grupos y software (N:N, 1:N y N:1). A diferencia de anteriores relaciones, en este caso no se puede filtrar por lista de técnicas disponibles. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna únicamente 3 df para las posibles relaciones 1:N, N:N y N:1 ya que en esta relación no hay disponibilidad de filtrar por técnica.
    '''
    # Verificamos que no haya habido algun error en la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Filtramos los objetos de tipo intrusion-set (grupos), tool y malware (software) y relationship (relaciones)
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # Creamos diccionarios para mapear los IDs y nombres
    group_id_map = {g['id']: next((ref['external_id'] for ref in g['external_references'] if ref['source_name'] == 'mitre-attack'), None) for g in groups}
    software_id_map = {s['id']: {
            'software_ID': next((ref['external_id'] for ref in s['external_references'] if ref['source_name'] == 'mitre-attack'), None),
            'software': s['name']
        } for s in software}

    # Creamos un diccionario para mapear los grupos y el software que usan
    group_software = []

    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'].startswith('intrusion-set') and rel['target_ref'].startswith(('tool--', 'malware--')):
            group_id = rel['source_ref']
            software_id = rel['target_ref']

            group_abbrev_id = group_id_map.get(group_id)
            software_info = software_id_map.get(software_id)
            group_name = next((g['name'] for g in groups if g['id'] == group_id), None)

            if group_abbrev_id and software_info and group_name:
                group_software.append({
                    'group_ID': group_abbrev_id,
                    'group': group_name,
                    'software_ID': software_info['software_ID'],
                    'software': software_info['software']
                })

    # Dataframe con los resultados
    MITRE_groups_software_NN_df = pd.DataFrame(group_software, columns=['group_ID', 'group', 'software_ID', 'software'])

    # Para finalizar agregamos por grupo y software con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por grupo y software's en una relación 1:N
    MITRE_group_software_1N_df = MITRE_groups_software_NN_df.groupby('group_ID', as_index=False).agg({
    'group':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })

    # Generamos el df compuesto por grupos y software en una relación N:1
    MITRE_groups_software_N1_df = MITRE_groups_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_groups_software_NN_df, MITRE_group_software_1N_df, MITRE_groups_software_N1_df

**Disponibilidad de reglas**

In [316]:
def get_availability_rules_by_technique(mitre_matrix_str):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica. Devuelve una primera tabla idéntica a la generada en el resumen generado en 'get_rules_and_classify_by_ttp/{matrix}-classified_rules.csv'. Y por otro lado genera una tabla con el agregado del número de reglas por técnica disponible que será requerida en posteriores cruces.
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix_str)
        CP_detail_rules_techniques = pd.DataFrame()

        if not os.path.exists(main_path):
            raise ValueError(f'No existe el fichero requerido: {main_path}, por favor ejecute el bloque get_rules_and_classify_by_ttp')

        file_path = os.path.join(main_path, f'{mitre_matrix_str}-classified_rules.csv')  # Corrección aquí
        CP_detail_rules_techniques = pd.read_csv(file_path, sep=';')
        CP_detail_rules_techniques = CP_detail_rules_techniques[CP_detail_rules_techniques['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
        CP_detail_rules_techniques = CP_detail_rules_techniques.sort_values(by='ttp')
        CP_detail_rules_techniques = CP_detail_rules_techniques.reset_index(drop=True)

        CP_agg_rules_techniques = CP_detail_rules_techniques[['ttp', 'rule', 'matrix']]
        CP_agg_rules_techniques = CP_agg_rules_techniques.groupby('ttp').agg(
            rules=('rule', 'count'),
            matrix=('matrix', 'first')
        ).sort_values(by='rules', ascending=False).reset_index()
 
    except ValueError as e:
        print(f'{e}')
        CP_detail_rules_techniques = None
        CP_agg_rules_techniques = None
    
    return CP_detail_rules_techniques, CP_agg_rules_techniques

In [317]:
def get_availability_rules_by_technique_and_tactics(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y táctica. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_tactics_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por táctica.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_tactics_NN_df,_,_,_,_,_,_,_,_ = get_techniques_tactics_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_tactics = pd.merge(CP_agg_rules_techniques, MITRE_techniques_tactics_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_tactics = CP_detail_rules_techniques_tactics[['tactic_ID', 'tactic', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('tactic_ID')
    CP_detail_rules_techniques_tactics = CP_detail_rules_techniques_tactics.reset_index(drop=True)
    CP_agg_rules_techniques_tactics = CP_detail_rules_techniques_tactics.groupby(['tactic_ID','tactic']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='tactic_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_tactics, CP_agg_rules_techniques_tactics

In [318]:
def get_availability_rules_by_technique_and_datasource(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y data source. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_datasources_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por data source.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_datasources_NN_df,_,_,_,_,_,_,_,_ = get_techniques_datasources_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_datasources = pd.merge(CP_agg_rules_techniques, MITRE_techniques_datasources_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_datasources = CP_detail_rules_techniques_datasources[['data_source_ID', 'data_source', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('data_source_ID')
    CP_detail_rules_techniques_datasources = CP_detail_rules_techniques_datasources.reset_index(drop=True)
    CP_agg_rules_techniques_datasources = CP_detail_rules_techniques_datasources.groupby(['data_source_ID','data_source']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='data_source_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_datasources, CP_agg_rules_techniques_datasources

In [319]:
def get_availability_rules_by_technique_and_platform(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y plataforma. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_platforms_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por plataforma.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_platforms_NN_df,_,_,_,_,_,_,_,_ = get_techniques_platforms_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_platforms = pd.merge(CP_agg_rules_techniques, MITRE_techniques_platforms_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_platforms = CP_detail_rules_techniques_platforms[['platform', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('platform')
    CP_detail_rules_techniques_platforms = CP_detail_rules_techniques_platforms.reset_index(drop=True)
    CP_agg_rules_techniques_platforms = CP_detail_rules_techniques_platforms.groupby(['platform']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='platform', ascending=True).reset_index()

    return CP_detail_rules_techniques_platforms, CP_agg_rules_techniques_platforms

In [320]:
def get_availability_rules_by_technique_and_group(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y grupo. Se llama a las funciones get_availability_rules_by_technique() y get_availability_rules_by_technique_and_group() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por grupo.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_groups_NN_df,_,_,_,_,_,_,_,_ = get_techniques_groups_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_groups = pd.merge(CP_agg_rules_techniques, MITRE_techniques_groups_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_groups = CP_detail_rules_techniques_groups[['group_ID', 'group', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('group_ID')
    CP_detail_rules_techniques_groups = CP_detail_rules_techniques_groups.reset_index(drop=True)
    CP_agg_rules_techniques_groups = CP_detail_rules_techniques_groups.groupby(['group_ID', 'group']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='group_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_groups, CP_agg_rules_techniques_groups

In [321]:
def get_availability_rules_by_technique_and_software(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y software. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_software_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por software.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_software_NN_df,_,_,_,_,_,_,_,_ = get_techniques_software_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_software = pd.merge(CP_agg_rules_techniques, MITRE_techniques_software_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_software = CP_detail_rules_techniques_software[['software_ID', 'software', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('software_ID')
    CP_detail_rules_techniques_software = CP_detail_rules_techniques_software.reset_index(drop=True)
    CP_agg_rules_techniques_software = CP_detail_rules_techniques_software.groupby(['software_ID', 'software']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='software_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_software, CP_agg_rules_techniques_software

### **Parámetros**

In [322]:
matrix = 'enterprise' # enterprise / ics / mobile
save_as_csv = True
debug_df = True # Parámetro para controlar el printeado de df

# **Ejecución principal**

## **1. Elementos generales**

### **1.1 Generación de la matriz MITRE**

In [323]:
mitre_matrix = get_data_from_branch(matrix)
mitre_matrix

### **1.2 Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada**

In [324]:
techniques_list = get_list_techniques_from_stix2(mitre_matrix,'both')
print(f"Se han generado la lista de técnicas (ID) para la matriz {matrix.upper()} que contiene un total de {len(techniques_list)} TTP's")

Se han generado la lista de técnicas (ID) para la matriz MOBILE que contiene un total de 119 TTP's


### **1.3 Obtención de las TTP disponbles en Cyber Proof con regla de detección**

In [325]:
cp_techniques =  get_CP_ttps_with_rules(matrix, way='file')
print(f"Se han obtenido un total de {len(cp_techniques)} TTP's con regla de detección asociada disponibles en CP.")

Se han obtenido un total de 15 TTP's con regla de detección asociada disponibles en CP.


## **2. Tablas informativas**

### **2.1. Técnicas**

#### **Generación de las tablas**

In [326]:
MITRE_techniques_df, CP_techniques_df, NOCP_techniques_df = get_techniques_information(mitre_matrix, cp_techniques, revoked_deprecated=True)

#### **Debug**

In [327]:
if debug_df:
    display(MITRE_techniques_df.head(3))
    display(CP_techniques_df.head(3))
    display(NOCP_techniques_df.head(3))
    print(f'{MITRE_techniques_df.shape}, {CP_techniques_df.shape}, {NOCP_techniques_df.shape}')

#### **Guardado**

In [328]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_df, f'[MITRE]_{matrix}_techniques',  matrix, 'information/techniques')
    save_df_as_csv(CP_techniques_df, f'[CP]_{matrix}_techniques', matrix, 'information/techniques')
    save_df_as_csv(NOCP_techniques_df, f'[NOCP]_{matrix}_techniques', matrix, 'information/techniques')

Archivo guardado correctamente [MITRE]_mobile_techniques.csv
Archivo guardado correctamente [CP]_mobile_techniques.csv
Archivo guardado correctamente [NOCP]_mobile_techniques.csv


### **2.2. Tácticas**

#### **Generación de la tabla**

In [329]:
MITRE_tactics_df = get_tactics_information(mitre_matrix)

#### **Debug**

In [330]:
if debug_df:
    display(MITRE_tactics_df.head(3))
    print(f'{MITRE_tactics_df.shape}')

#### **Guardado**

In [331]:
if save_as_csv:
    save_df_as_csv(MITRE_tactics_df, f'[MITRE]_{matrix}_tactics', matrix, 'information/tactics')

Archivo guardado correctamente [MITRE]_mobile_tactics.csv


### **2.3. Data sources**

#### **Generación de las tablas**

In [332]:
MITRE_datasources_df = get_datasources_information(mitre_matrix, False)

#### **Debug**

In [333]:
if debug_df:
    display(MITRE_datasources_df.head(3))
    print(f'{MITRE_datasources_df.shape}')

#### **Guardado**

In [334]:
if save_as_csv:
    save_df_as_csv(MITRE_datasources_df, f'[MITRE]_{matrix}_datasources', matrix, 'information/datasources')

Archivo guardado correctamente [MITRE]_mobile_datasources.csv


### **2.4. Plataformas**

#### **Generación de las tablas**

In [335]:
MITRE_platforms_df = get_platforms_information(mitre_matrix)

#### **Debug**

In [336]:
if debug_df:
    display(MITRE_platforms_df.head(3))
    print(f'{MITRE_platforms_df.shape}')

#### **Guardado**

In [337]:
if save_as_csv:
    save_df_as_csv(MITRE_platforms_df, f'[MITRE]_{matrix}_platforms', matrix, 'information/platforms')


Archivo guardado correctamente [MITRE]_mobile_platforms.csv


### **2.5. Grupos**

#### **Generación de las tablas**

In [338]:
MITRE_groups_df = get_groups_information(mitre_matrix, revoked_deprecated=True)

#### **Debug**

In [339]:
if debug_df:
    display(MITRE_groups_df.head(3))
    print(f'{MITRE_groups_df.shape}')

#### **Guardado**

In [340]:
if save_as_csv:
    save_df_as_csv(MITRE_groups_df, f'[MITRE]_{matrix}_groups', matrix, 'information/groups')

Archivo guardado correctamente [MITRE]_mobile_groups.csv


### **2.6. Software**

#### **Generación de las tablas**

In [341]:
MITRE_software_df = get_software_information(mitre_matrix, revoked_deprecated=True)

#### **Debug**

In [342]:
if debug_df:
    display(MITRE_software_df.head(3))
    print(f'{MITRE_software_df.shape}')

#### **Guardado**

In [343]:
if save_as_csv:
    save_df_as_csv(MITRE_groups_df, f'[MITRE]_{matrix}_software', matrix, 'information/software')

Archivo guardado correctamente [MITRE]_mobile_software.csv


## **3. Relaciones**

### **3.1. Relación técnicas - tácticas**

#### **Generación de las tablas**

In [344]:
MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df = get_techniques_tactics_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [345]:
if debug_df:
    display(MITRE_techniques_tactics_NN_df.head(3))
    display(CP_techniques_tactics_NN_df.head(3))
    display(NOCP_techniques_tactics_NN_df.head(3))
    print(f'{MITRE_techniques_tactics_NN_df.shape}, {CP_techniques_tactics_NN_df.shape}, {NOCP_techniques_tactics_NN_df.shape}')

In [346]:
if debug_df:
    display(MITRE_technique_tactics_1N_df.head(3))
    display(CP_technique_tactics_1N_df.head(3))
    display(NOCP_technique_tactics_1N_df.head(3))
    print(f'{MITRE_technique_tactics_1N_df.shape}, {CP_technique_tactics_1N_df.shape}, {NOCP_technique_tactics_1N_df.shape}')

In [347]:
if debug_df:
    display(MITRE_techniques_tactic_N1_df.head(3))
    display(CP_techniques_tactic_N1_df.head(3))
    display(NOCP_techniques_tactic_N1_df.head(3))
    print(f'{MITRE_techniques_tactic_N1_df.shape}, {CP_techniques_tactic_N1_df.shape}, {NOCP_techniques_tactic_N1_df.shape}')

#### **Guardado**

In [348]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_tactics_NN_df, f'[MITRE]_{matrix}_techniques_tactics_NN', matrix, 'relations/techniques_tactics')
    save_df_as_csv(CP_techniques_tactics_NN_df, f'[CP]_{matrix}_techniques_tactics_NN', matrix, 'relations/techniques_tactics')
    save_df_as_csv(NOCP_techniques_tactics_NN_df, f'[NOCP]_{matrix}_techniques_tactics_NN', matrix, 'relations/techniques_tactics')

    save_df_as_csv(MITRE_technique_tactics_1N_df, f'[MITRE]_{matrix}_technique_tactics_1N', matrix, 'relations/techniques_tactics')
    save_df_as_csv(CP_technique_tactics_1N_df, f'[CP]_{matrix}_technique_tactics_1N', matrix, 'relations/techniques_tactics')
    save_df_as_csv(NOCP_technique_tactics_1N_df, f'[NOCP]_{matrix}_technique_tactics_1N', matrix, 'relations/techniques_tactics')

    save_df_as_csv(MITRE_techniques_tactic_N1_df, f'[MITRE]_{matrix}_techniques_tactic_N1', matrix, 'relations/techniques_tactics')
    save_df_as_csv(CP_techniques_tactic_N1_df, f'[CP]_{matrix}_techniques_tactic_N1', matrix, 'relations/techniques_tactics')
    save_df_as_csv(NOCP_techniques_tactic_N1_df, f'[NOCP]_{matrix}_techniques_tactic_N1', matrix, 'relations/techniques_tactics')

Archivo guardado correctamente [MITRE]_mobile_techniques_tactics_NN.csv
Archivo guardado correctamente [CP]_mobile_techniques_tactics_NN.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_tactics_NN.csv
Archivo guardado correctamente [MITRE]_mobile_technique_tactics_1N.csv
Archivo guardado correctamente [CP]_mobile_technique_tactics_1N.csv
Archivo guardado correctamente [NOCP]_mobile_technique_tactics_1N.csv
Archivo guardado correctamente [MITRE]_mobile_techniques_tactic_N1.csv
Archivo guardado correctamente [CP]_mobile_techniques_tactic_N1.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_tactic_N1.csv


### **3.2. Relación técnicas - data sources**

#### **Generación de las tablas**

In [349]:
MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_datasource_N1_df, CP_techniques_datasource_N1_df, NOCP_techniques_datasource_N1_df = get_techniques_datasources_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [350]:
if debug_df:
    display(MITRE_techniques_datasources_NN_df.head(3))
    display(CP_techniques_datasources_NN_df.head(3))
    display(NOCP_techniques_datasources_NN_df.head(3))
    print(f'{MITRE_techniques_datasources_NN_df.shape}, {CP_techniques_datasources_NN_df.shape}, {NOCP_techniques_datasources_NN_df.shape}')

In [351]:
if debug_df:
    display(MITRE_technique_datasources_1N_df.head(3))
    display(CP_technique_datasources_1N_df.head(3))
    display(NOCP_technique_datasources_1N_df.head(3))
    print(f'{MITRE_technique_datasources_1N_df.shape}, {CP_technique_datasources_1N_df.shape}, {NOCP_technique_datasources_1N_df.shape}')

In [352]:
if debug_df:
    display(MITRE_techniques_datasource_N1_df.head(3))
    display(CP_techniques_datasource_N1_df.head(3))
    display(NOCP_techniques_datasource_N1_df.head(3))
    print(f'{MITRE_techniques_datasource_N1_df.shape}, {CP_techniques_datasource_N1_df.shape}, {NOCP_techniques_datasource_N1_df.shape}')

#### **Guardado**

In [353]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_datasources_NN_df, f'[MITRE]_{matrix}_techniques_datasources_NN', matrix, 'relations/techniques_datasources')
    save_df_as_csv(CP_techniques_datasources_NN_df, f'[CP]_{matrix}_techniques_datasources_NN', matrix, 'relations/techniques_datasources')
    save_df_as_csv(NOCP_techniques_datasources_NN_df, f'[NOCP]_{matrix}_techniques_datasources_NN', matrix, 'relations/techniques_datasources')

    save_df_as_csv(MITRE_technique_datasources_1N_df, f'[MITRE]_{matrix}_technique_datasources_1N', matrix, 'relations/techniques_datasources')
    save_df_as_csv(CP_technique_datasources_1N_df, f'[CP]_{matrix}_technique_datasources_1N', matrix, 'relations/techniques_datasources')
    save_df_as_csv(NOCP_technique_datasources_1N_df, f'[NOCP]_{matrix}_technique_datasources_1N', matrix, 'relations/techniques_datasources')

    save_df_as_csv(MITRE_techniques_datasource_N1_df, f'[MITRE]_{matrix}_techniques_datasource_N1', matrix, 'relations/techniques_datasources')
    save_df_as_csv(CP_techniques_datasource_N1_df, f'[CP]_{matrix}_techniques_datasource_N1', matrix, 'relations/techniques_datasources')
    save_df_as_csv(NOCP_techniques_datasource_N1_df, f'[NOCP]_{matrix}_techniques_datasource_N1', matrix, 'relations/techniques_datasources')

Archivo guardado correctamente [MITRE]_mobile_techniques_datasources_NN.csv
Archivo guardado correctamente [CP]_mobile_techniques_datasources_NN.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_datasources_NN.csv
Archivo guardado correctamente [MITRE]_mobile_technique_datasources_1N.csv
Archivo guardado correctamente [CP]_mobile_technique_datasources_1N.csv
Archivo guardado correctamente [NOCP]_mobile_technique_datasources_1N.csv
Archivo guardado correctamente [MITRE]_mobile_techniques_datasource_N1.csv
Archivo guardado correctamente [CP]_mobile_techniques_datasource_N1.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_datasource_N1.csv


### **3.3. Relación técnicas - plataformas**

#### **Generación de las tablas**

In [354]:
MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df = get_techniques_platforms_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [355]:
if debug_df:
    display(MITRE_techniques_platforms_NN_df.head(3))
    display(CP_techniques_platforms_NN_df.head(3))
    display(NOCP_techniques_platforms_NN_df.head(3))
    print(f'{MITRE_techniques_platforms_NN_df.shape}, {CP_techniques_platforms_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

In [356]:
if debug_df:
    display(MITRE_technique_platforms_1N_df.head(3))
    display(CP_technique_platforms_1N_df.head(3))
    display(NOCP_technique_platforms_1N_df.head(3))
    print(f'{MITRE_technique_platforms_1N_df.shape}, {CP_technique_platforms_1N_df.shape}, {NOCP_technique_platforms_1N_df.shape}')

In [357]:
if debug_df:
    display(MITRE_techniques_platform_N1_df.head(3))
    display(CP_techniques_platform_N1_df.head(3))
    display(NOCP_techniques_platform_N1_df.head(3))
    print(f'{MITRE_techniques_platform_N1_df.shape}, {CP_techniques_platform_N1_df.shape}, {NOCP_techniques_platform_N1_df.shape}')

#### **Guardado**

In [358]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_platforms_NN_df, f'[MITRE]_{matrix}_techniques_platforms_NN', matrix, 'relations/techniques_platforms')
    save_df_as_csv(CP_techniques_platforms_NN_df, f'[CP]_{matrix}_techniques_platforms_NN', matrix, 'relations/techniques_platforms')
    save_df_as_csv(NOCP_techniques_platforms_NN_df, f'[NOCP]_{matrix}_techniques_platforms_NN', matrix, 'relations/techniques_platforms')

    save_df_as_csv(MITRE_technique_platforms_1N_df, f'[MITRE]_{matrix}_technique_platforms_1N', matrix, 'relations/techniques_platforms')
    save_df_as_csv(CP_technique_platforms_1N_df, f'[CP]_{matrix}_technique_platforms_1N', matrix, 'relations/techniques_platforms')
    save_df_as_csv(NOCP_technique_platforms_1N_df, f'[NOCP]_{matrix}_technique_platforms_1N', matrix, 'relations/techniques_platforms')

    save_df_as_csv(MITRE_techniques_platform_N1_df, f'[MITRE]_{matrix}_techniques_platform_N1', matrix, 'relations/techniques_platforms')
    save_df_as_csv(CP_techniques_platform_N1_df, f'[CP]_{matrix}_techniques_platform_N1', matrix, 'relations/techniques_platforms')
    save_df_as_csv(NOCP_techniques_platform_N1_df, f'[NOCP]_{matrix}_techniques_platform_N1', matrix, 'relations/techniques_platforms')

Archivo guardado correctamente [MITRE]_mobile_techniques_platforms_NN.csv
Archivo guardado correctamente [CP]_mobile_techniques_platforms_NN.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_platforms_NN.csv
Archivo guardado correctamente [MITRE]_mobile_technique_platforms_1N.csv
Archivo guardado correctamente [CP]_mobile_technique_platforms_1N.csv
Archivo guardado correctamente [NOCP]_mobile_technique_platforms_1N.csv
Archivo guardado correctamente [MITRE]_mobile_techniques_platform_N1.csv
Archivo guardado correctamente [CP]_mobile_techniques_platform_N1.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_platform_N1.csv


### **3.4. Relación técnicas - grupos**

#### **Generación de las tablas**

In [359]:
MITRE_techniques_groups_NN_df, CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df, MITRE_technique_groups_1N_df, CP_technique_groups_1N_df, NOCP_technique_groups_1N_df, MITRE_techniques_group_N1_df, CP_techniques_group_N1_df, NOCP_techniques_group_N1_df = get_techniques_groups_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [360]:
if debug_df:
    display(MITRE_techniques_groups_NN_df.head(3))
    display(CP_techniques_groups_NN_df.head(3))
    display(NOCP_techniques_groups_NN_df.head(3))
    print(f'{MITRE_techniques_groups_NN_df.shape}, {CP_techniques_groups_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

In [361]:
if debug_df:
    display(MITRE_technique_groups_1N_df.head(3))
    display(CP_technique_groups_1N_df.head(3))
    display(NOCP_technique_groups_1N_df.head(3))
    print(f'{MITRE_technique_groups_1N_df.shape}, {CP_technique_groups_1N_df.shape}, {NOCP_technique_groups_1N_df.shape}')

In [362]:
if debug_df:
    display(MITRE_techniques_group_N1_df.head(3))
    display(CP_techniques_group_N1_df.head(3))
    display(NOCP_techniques_group_N1_df.head(3))
    print(f'{MITRE_techniques_group_N1_df.shape}, {CP_techniques_group_N1_df.shape}, {NOCP_techniques_group_N1_df.shape}')

#### **Guardado**

In [363]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_groups_NN_df, f'[MITRE]_{matrix}_techniques_groups_NN', matrix, 'relations/techniques_groups')
    save_df_as_csv(CP_techniques_groups_NN_df, f'[CP]_{matrix}_techniques_groups_NN', matrix, 'relations/techniques_groups')
    save_df_as_csv(NOCP_techniques_groups_NN_df, f'[NOCP]_{matrix}_techniques_groups_NN', matrix, 'relations/techniques_groups')

    save_df_as_csv(MITRE_technique_groups_1N_df, f'[MITRE]_{matrix}_technique_groups_1N', matrix, 'relations/techniques_groups')
    save_df_as_csv(CP_technique_groups_1N_df, f'[CP]_{matrix}_technique_groups_1N', matrix, 'relations/techniques_groups')
    save_df_as_csv(NOCP_technique_groups_1N_df, f'[NOCP]_{matrix}_technique_groups_1N', matrix, 'relations/techniques_groups')

    save_df_as_csv(MITRE_techniques_group_N1_df, f'[MITRE]_{matrix}_techniques_group_N1', matrix, 'relations/techniques_groups')
    save_df_as_csv(CP_techniques_group_N1_df, f'[CP]_{matrix}_techniques_group_N1', matrix, 'relations/techniques_groups')
    save_df_as_csv(NOCP_techniques_group_N1_df, f'[NOCP]_{matrix}_techniques_group_N1', matrix, 'relations/techniques_groups')

Archivo guardado correctamente [MITRE]_mobile_techniques_groups_NN.csv
Archivo guardado correctamente [CP]_mobile_techniques_groups_NN.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_groups_NN.csv
Archivo guardado correctamente [MITRE]_mobile_technique_groups_1N.csv
Archivo guardado correctamente [CP]_mobile_technique_groups_1N.csv
Archivo guardado correctamente [NOCP]_mobile_technique_groups_1N.csv
Archivo guardado correctamente [MITRE]_mobile_techniques_group_N1.csv
Archivo guardado correctamente [CP]_mobile_techniques_group_N1.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_group_N1.csv


### **3.5. Relación técnicas - software**

#### **Generación de las tablas**

In [364]:
MITRE_techniques_software_NN_df, CP_techniques_software_NN_df, NOCP_techniques_software_NN_df, MITRE_technique_software_1N_df, CP_technique_software_1N_df, NOCP_technique_software_1N_df, MITRE_techniques_software_N1_df, CP_techniques_software_N1_df, NOCP_techniques_software_N1_df = get_techniques_software_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [365]:
if debug_df:
    display(MITRE_techniques_software_NN_df.head(3))
    display(CP_techniques_software_NN_df.head(3))
    display(NOCP_techniques_software_NN_df.head(3))
    print(f'{MITRE_techniques_software_NN_df.shape}, {CP_techniques_software_NN_df.shape}, {NOCP_techniques_software_NN_df.shape}')

In [366]:
if debug_df:
    display(MITRE_technique_software_1N_df.head(3))
    display(CP_technique_software_1N_df.head(3))
    display(NOCP_technique_software_1N_df.head(3))
    print(f'{MITRE_technique_software_1N_df.shape}, {CP_technique_software_1N_df.shape}, {NOCP_technique_software_1N_df.shape}')

In [367]:
if debug_df:
    display(MITRE_techniques_software_N1_df.head(3))
    display(CP_techniques_software_N1_df.head(3))
    display(NOCP_techniques_software_N1_df.head(3))
    print(f'{MITRE_techniques_software_N1_df.shape}, {CP_techniques_software_N1_df.shape}, {NOCP_techniques_software_N1_df.shape}')

#### **Guardado**

In [368]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_software_NN_df, f'[MITRE]_{matrix}_techniques_software_NN', matrix, 'relations/techniques_software')
    save_df_as_csv(CP_techniques_software_NN_df, f'[CP]_{matrix}_techniques_software_NN', matrix, 'relations/techniques_software')
    save_df_as_csv(NOCP_techniques_software_NN_df, f'[NOCP]_{matrix}_techniques_software_NN', matrix, 'relations/techniques_software')

    save_df_as_csv(MITRE_technique_software_1N_df, f'[MITRE]_{matrix}_technique_software_1N', matrix, 'relations/techniques_software')
    save_df_as_csv(CP_technique_software_1N_df, f'[CP]_{matrix}_technique_software_1N', matrix, 'relations/techniques_software')
    save_df_as_csv(NOCP_technique_software_1N_df, f'[NOCP]_{matrix}_technique_software_1N', matrix, 'relations/techniques_software')

    save_df_as_csv(MITRE_techniques_software_N1_df, f'[MITRE]_{matrix}_techniques_software_N1', matrix, 'relations/techniques_software')
    save_df_as_csv(CP_techniques_software_N1_df, f'[CP]_{matrix}_techniques_software_N1', matrix, 'relations/techniques_software')
    save_df_as_csv(NOCP_techniques_software_N1_df, f'[NOCP]_{matrix}_techniques_software_N1', matrix, 'relations/techniques_software')

Archivo guardado correctamente [MITRE]_mobile_techniques_software_NN.csv
Archivo guardado correctamente [CP]_mobile_techniques_software_NN.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_software_NN.csv
Archivo guardado correctamente [MITRE]_mobile_technique_software_1N.csv
Archivo guardado correctamente [CP]_mobile_technique_software_1N.csv
Archivo guardado correctamente [NOCP]_mobile_technique_software_1N.csv
Archivo guardado correctamente [MITRE]_mobile_techniques_software_N1.csv
Archivo guardado correctamente [CP]_mobile_techniques_software_N1.csv
Archivo guardado correctamente [NOCP]_mobile_techniques_software_N1.csv


### **3.6. Relación grupos - software**

#### **Generación de las tablas**

In [369]:
MITRE_groups_software_NN_df, MITRE_group_software_1N_df, MITRE_groups_software_N1_df = get_groups_software_relationships(mitre_matrix)

Dataframes generados correctamente!


#### **Debug**

In [370]:
if debug_df:
    display(MITRE_groups_software_NN_df.head(3))
    display(MITRE_group_software_1N_df.head(3))
    display(MITRE_groups_software_N1_df.head(3))
    print(f'{MITRE_groups_software_NN_df.shape}, {MITRE_group_software_1N_df.shape}, {MITRE_groups_software_N1_df.shape}')

#### **Guardado**

In [371]:
if save_as_csv:
    save_df_as_csv(MITRE_groups_software_NN_df, f'[MITRE]_{matrix}_groups_software_NN', matrix, 'relations/groups_software')
    save_df_as_csv(MITRE_group_software_1N_df, f'[MITRE]_{matrix}_group_software_1N', matrix, 'relations/groups_software')
    save_df_as_csv(MITRE_groups_software_N1_df, f'[MITRE]_{matrix}_groups_software_N1', matrix, 'relations/groups_software')

Archivo guardado correctamente [MITRE]_mobile_groups_software_NN.csv
Archivo guardado correctamente [MITRE]_mobile_group_software_1N.csv
Archivo guardado correctamente [MITRE]_mobile_groups_software_N1.csv


## **4. Disponibilidad de reglas por tipología**

### **4.1. Reglas disponibles por técnica**

In [372]:
CP_detail_rules_techniques, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix)

In [373]:
if debug_df:
    display(CP_detail_rules_techniques.head(3))
    display(CP_agg_rules_techniques.head(3))
    print(f'{CP_detail_rules_techniques.shape}, {CP_agg_rules_techniques.shape}')

In [374]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques, f'{matrix}_detail_rules_techniques', matrix, 'availability_rules/techniques')
    save_df_as_csv(CP_agg_rules_techniques, f'{matrix}_agg_rules_techniques', matrix, 'availability_rules/techniques')

Archivo guardado correctamente mobile_detail_rules_techniques.csv
Archivo guardado correctamente mobile_agg_rules_techniques.csv


### **4.2. Reglas disponibles por técnica y táctica**

In [375]:
CP_detail_rules_techniques_tactics, CP_agg_rules_techniques_tactics = get_availability_rules_by_technique_and_tactics(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [376]:
if debug_df:
    display(CP_detail_rules_techniques_tactics.head(3))
    display(CP_agg_rules_techniques_tactics.head(3))
    print(f'{CP_detail_rules_techniques_tactics.shape}, {CP_agg_rules_techniques_tactics.shape}')

In [377]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_tactics, f'{matrix}_detail_rules_techniques_tactics', matrix, 'availability_rules/techniques_tactics')
    save_df_as_csv(CP_agg_rules_techniques_tactics, f'{matrix}_agg_rules_techniques_tactics', matrix, 'availability_rules/techniques_tactics')

Archivo guardado correctamente mobile_detail_rules_techniques_tactics.csv
Archivo guardado correctamente mobile_agg_rules_techniques_tactics.csv


### **4.3. Reglas disponibles por técnica y data source**

In [378]:
CP_detail_rules_techniques_datasources, CP_agg_rules_techniques_datasources = get_availability_rules_by_technique_and_datasource(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [379]:
if debug_df:
    display(CP_detail_rules_techniques_datasources.head(3))
    display(CP_agg_rules_techniques_datasources.head(3))
    print(f'{CP_detail_rules_techniques_datasources.shape}, {CP_agg_rules_techniques_datasources.shape}')

In [380]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_datasources, f'{matrix}_detail_rules_techniques_datasources', matrix, 'availability_rules/techniques_datasources')
    save_df_as_csv(CP_agg_rules_techniques_datasources, f'{matrix}_agg_rules_techniques_datasources', matrix, 'availability_rules/techniques_datasources')

Archivo guardado correctamente mobile_detail_rules_techniques_datasources.csv
Archivo guardado correctamente mobile_agg_rules_techniques_datasources.csv


### **4.4. Reglas disponibles por técnica y plataforma**

In [381]:
CP_detail_rules_techniques_platforms, CP_agg_rules_techniques_platforms = get_availability_rules_by_technique_and_platform(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [382]:
if debug_df:
    display(CP_detail_rules_techniques_platforms.head(3))
    display(CP_agg_rules_techniques_platforms.head(3))
    print(f'{CP_detail_rules_techniques_platforms.shape}, {CP_agg_rules_techniques_platforms.shape}')

In [383]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_platforms, f'{matrix}_detail_rules_techniques_platforms', matrix, 'availability_rules/techniques_platforms')
    save_df_as_csv(CP_agg_rules_techniques_platforms, f'{matrix}_agg_rules_techniques_platforms', matrix, 'availability_rules/techniques_platforms')

Archivo guardado correctamente mobile_detail_rules_techniques_platforms.csv
Archivo guardado correctamente mobile_agg_rules_techniques_platforms.csv


### **4.5. Reglas disponibles por técnica y grupo**

In [384]:
CP_detail_rules_techniques_groups, CP_agg_rules_techniques_groups = get_availability_rules_by_technique_and_group(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [385]:
if debug_df:
    display(CP_detail_rules_techniques_groups.head(3))
    display(CP_agg_rules_techniques_groups.head(3))
    print(f'{CP_detail_rules_techniques_groups.shape}, {CP_agg_rules_techniques_groups.shape}')

In [386]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_groups, f'{matrix}_detail_rules_techniques_groups', matrix, 'availability_rules/techniques_groups')
    save_df_as_csv(CP_agg_rules_techniques_groups, f'{matrix}_agg_rules_techniques_groups', matrix, 'availability_rules/techniques_groups')

Archivo guardado correctamente mobile_detail_rules_techniques_groups.csv
Archivo guardado correctamente mobile_agg_rules_techniques_groups.csv


### **4.6. Reglas disponibles por técnica y software**

In [387]:
CP_detail_rules_techniques_software, CP_agg_rules_techniques_software = get_availability_rules_by_technique_and_software(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [388]:
if debug_df:
    display(CP_detail_rules_techniques_software.head(3))
    display(CP_agg_rules_techniques_software.head(3))
    print(f'{CP_detail_rules_techniques_software.shape}, {CP_agg_rules_techniques_software.shape}')

In [389]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_software, f'{matrix}_detail_rules_techniques_software', matrix, 'availability_rules/techniques_software')
    save_df_as_csv(CP_agg_rules_techniques_software, f'{matrix}_agg_rules_techniques_software', matrix, 'availability_rules/techniques_software')

Archivo guardado correctamente mobile_detail_rules_techniques_software.csv
Archivo guardado correctamente mobile_agg_rules_techniques_software.csv
